# BERT Yelp Sentiment Analysis

In [33]:
# Importing neccessary libraries

# Data manipulation & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Text preprocessing
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

# Dataset handling
from datasets import Dataset, load_from_disk
from imblearn.under_sampling import RandomUnderSampler

# Hugging Face
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

# LoRA
from peft import LoraConfig, get_peft_model

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/yevheniiakysel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/yevheniiakysel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [34]:
# Load the cleaned CSV
reviews_df = pd.read_csv('../data/processed/yelp_reviews_cleaned.csv')
print(f"Loaded {len(reviews_df)} rows from CSV\n")
print(reviews_df.info())

Loaded 99966 rows from CSV

<class 'pandas.DataFrame'>
RangeIndex: 99966 entries, 0 to 99965
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   stars            99966 non-null  int64  
 1   text             99966 non-null  str    
 2   cleaned_text     99962 non-null  str    
 3   sentiment_score  99966 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 88.3 MB
None


In [35]:
# Map the sentiment labels (positive, negative and neutral) to corresponding star ratings in the DataFrame for training
reviews_df['label'] = reviews_df['stars'].replace({
    1: 0, # negative
    2: 0, 
    3: 1, # neutral
    4: 2,
    5: 2  # positive
})

# Add a column with explicit label names for readability
reviews_df['label_names'] = reviews_df['label'].replace({
    0: "NEGATIVE",
    1: "NEUTRAL",
    2: "POSITIVE",
})

In [40]:
# Check for class imbalance in the dataset
print(f"Distribution of labels in the dataset:\n{reviews_df['label'].value_counts().sort_index()}")

# Define the data and target features
X = reviews_df[['cleaned_text']] # Convert to a DataFrame as RandomUnderSampler expects 2D features
y = reviews_df['label']

undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=42) # Resample all classes but the minority class, controlling randomization reproducibility with the magical 42 ("Answer to the Ultimate Question of Life")

X_res, y_res = undersampler.fit_resample(X, y)

print(f"Before undersampling: {Counter(y)}")
print(f"Before undersampling: {Counter(y_res)}")

balanced_reviews_df = pd.DataFrame({
    'cleaned_text': X_res['cleaned_text'], # Convert from 2D DataFrame to 1D Series
    'label': y_res
})

print(balanced_reviews_df.sort_index().head())

Distribution of labels in the dataset:
label
0    18902
1    11361
2    69703
Name: count, dtype: int64
Before undersampling: Counter({2: 69703, 0: 18902, 1: 11361})
Before undersampling: Counter({0: 11361, 1: 11361, 2: 11361})
                                        cleaned_text  label
0  decide eat aware going take hours beginning en...      1
2  family diner buffet eclectic assortment large ...      1
5  long term frequent customer establishment went...      0
8  easter instead going lopez lake went los padre...      1
9  party hibachi waitress brought separate sushi ...      1


In [37]:
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"} # A map from index to label
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2} # A map from label to index

In [44]:
# Add mapped label names for readability
balanced_reviews_df['label_names'] = balanced_reviews_df['label'].replace(label2id)

# Convert the balanced DataFrame to a Hugging Face Dataset object
balanced_reviews_hf = Dataset.from_pandas(balanced_reviews_df)

# Save the HuggingFace Dataset to disk
balanced_reviews_hf.save_to_disk('../data/processed/yelp_reviews_balanced_dataset')

Saving the dataset (0/1 shards):   0%|          | 0/34083 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=3, id2label=id2label, label2id=label2id)

In [ ]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert_base_uncased")